# Experiment 2: Discogs API Exploration

**Goal:** Understand what Discogs adds over MusicBrainz — sub-genres (`styles`), artist biographies (`profile`), and release format data.

**Key questions:**
- What does the `profile` (biography text) look like? Useful as a text source?
- How do `genres` vs `styles` compare to MusicBrainz tags?
- Can we extract the Discogs artist ID from MusicBrainz URL rels and use it directly?
- What additional fields does Discogs offer that MusicBrainz doesn't?

**Rate limit:** 25 req/min unauthenticated (with User-Agent).

## Setup

In [1]:
import requests
import json
import time

DISCOGS_BASE = "https://api.discogs.com"
HEADERS = {
    "User-Agent": "KE-CW2-MusicHistory/0.1"
}

def discogs_get(endpoint, params=None):
    """Helper to make Discogs API requests with rate limiting."""
    url = f"{DISCOGS_BASE}{endpoint}"
    resp = requests.get(url, headers=HEADERS, params=params)
    if resp.status_code == 429:
        print("Rate limited! Waiting 60s...")
        time.sleep(60)
        resp = requests.get(url, headers=HEADERS, params=params)
    resp.raise_for_status()
    return resp.json()

def pp(data):
    print(json.dumps(data, indent=2, default=str))

## 1. Fetch an Artist by Discogs ID

From Experiment 1, we know MusicBrainz links Bowie to `https://www.discogs.com/artist/10263`.

Let's fetch that directly.

In [2]:
# David Bowie's Discogs ID (from MusicBrainz url-rels)
bowie_discogs_id = 10263

bowie = discogs_get(f"/artists/{bowie_discogs_id}")
pp(bowie)

{
  "name": "David Bowie",
  "id": 10263,
  "resource_url": "https://api.discogs.com/artists/10263",
  "uri": "https://www.discogs.com/artist/10263-David-Bowie",
  "releases_url": "https://api.discogs.com/artists/10263/releases",
  "images": [
    {
      "type": "primary",
      "uri": "https://i.discogs.com/Nc1FSm4J27mP2ZdvaR9uluSd7T8Vhn6uf5jw2SHdI3I/rs:fit/g:sm/q:90/h:591/w:598/czM6Ly9kaXNjb2dz/LWRhdGFiYXNlLWlt/YWdlcy9BLTEwMjYz/LTEzNDA4NzIyMTMt/OTc4MC5qcGVn.jpeg",
      "resource_url": "https://i.discogs.com/Nc1FSm4J27mP2ZdvaR9uluSd7T8Vhn6uf5jw2SHdI3I/rs:fit/g:sm/q:90/h:591/w:598/czM6Ly9kaXNjb2dz/LWRhdGFiYXNlLWlt/YWdlcy9BLTEwMjYz/LTEzNDA4NzIyMTMt/OTc4MC5qcGVn.jpeg",
      "uri150": "https://i.discogs.com/Yc9NdKTWHMcmDaJeEDuaAIBpZTLnGR4HHm16rfbd00Q/rs:fit/g:sm/q:40/h:150/w:150/czM6Ly9kaXNjb2dz/LWRhdGFiYXNlLWlt/YWdlcy9BLTEwMjYz/LTEzNDA4NzIyMTMt/OTc4MC5qcGVn.jpeg",
      "width": 598,
      "height": 591
    },
    {
      "type": "secondary",
      "uri": "https://i.discogs.com/-lpIA9

### 1a. Basic Info

Compare with MusicBrainz fields.

In [3]:
print(f"Name:        {bowie.get('name', 'N/A')}")
print(f"Real Name:   {bowie.get('realname', 'N/A')}")
print(f"Discogs ID:  {bowie.get('id', 'N/A')}")
print(f"Data Quality: {bowie.get('data_quality', 'N/A')}")

# Namevariations — aliases
name_vars = bowie.get('namevariations', [])
print(f"\nName Variations ({len(name_vars)}):")
for nv in name_vars[:10]:
    print(f"  - {nv}")

# URLs
urls = bowie.get('urls', [])
print(f"\nURLs ({len(urls)}):")
for url in urls:
    print(f"  - {url}")

Name:        David Bowie
Real Name:   David Robert Jones
Discogs ID:  10263
Data Quality: Needs Vote

Name Variations (56):
  - 2-1
  - B. Bowie
  - Boh Oui
  - Bowie
  - Bowie D.
  - Bowie David
  - Bowie!
  - Bowie, David
  - D .Bowie
  - D Bowie

URLs (21):
  - https://www.davidbowie.com
  - https://www.facebook.com/davidbowie
  - https://www.youtube.com/davidbowie
  - https://www.instagram.com/davidbowie/
  - https://www.last.fm/music/David+Bowie
  - https://soundcloud.com/davidbowieofficial
  - https://en.wikipedia.org/wiki/David_Bowie
  - https://www.youtube.com/user/DavidBowieVEVO
  - https://www.biography.com/musician/david-bowie
  - https://bowieontape.com/
  - https://bowiesongs.wordpress.com/
  - https://www.bowiewonderworld.com/
  - https://www.davidbowienews.com/
  - https://www.geni.com/people/David-Bowie/6000000008762972867
  - https://www.illustrated-db-discography.nl/
  - https://www.imdb.com/name/nm0000309/
  - https://www.britannica.com/biography/David-Bowie
  - http

### 1b. Profile (Biography Text)

This is a **textual data source** — free text biography that could be used for NER/LLM extraction.
Compare with Wikipedia article text later.

In [4]:
profile = bowie.get('profile', 'No profile')
print(f"Profile length: {len(profile)} characters\n")
print(profile[:2000])
if len(profile) > 2000:
    print(f"\n... [{len(profile) - 2000} more characters]")

Profile length: 524 characters

British pop/rock singer, musician, songwriter, and actor.

Born: 8 January 1947 in Brixton, London, England, UK.
Died: 10 January 2016 in Manhattan, New York City, USA (aged 69).

Bowie is recognized as one of the most respected contemporary musicians of his period. He was a leading figure in the music industry and is considered one of the most influential musicians of the 20th century.
Inducted into Rock And Roll Hall of Fame in 1996.

For a list of all band and group involvement, please see [b][a1240431][/b].


### 1c. Members / Groups

Discogs tracks which groups an artist was a member of.

In [5]:
groups = bowie.get('groups', [])
print(f"Groups ({len(groups)}):")
for g in groups:
    print(f"  - {g['name']} (ID: {g['id']}, active: {g.get('active', '?')})")

Groups (0):


## 2. Artist Releases — Genres vs Styles

In Discogs, **genres** are broad categories and **styles** are sub-genres.
This maps to our `hasGenre` and `subgenreOf` properties.

In [6]:
releases = discogs_get(f"/artists/{bowie_discogs_id}/releases", params={"per_page": 10})

print(f"Total releases: {releases['pagination']['items']}")
print(f"Showing first {len(releases['releases'])}:\n")

for rel in releases['releases']:
    print(f"  {rel.get('title', 'N/A')}")
    print(f"    Year:   {rel.get('year', 'N/A')}")
    print(f"    Type:   {rel.get('type', 'N/A')}")
    print(f"    Role:   {rel.get('role', 'N/A')}")
    print(f"    Format: {rel.get('format', 'N/A')}")
    print(f"    Label:  {rel.get('label', 'N/A')}")
    print()

Total releases: 10629
Showing first 10:

  I Do Believe I Love You
    Year:   1966
    Type:   release
    Role:   Main
    Format: Acetate, 7", S/Sided, Mono
    Label:  Orbit Music (2)

  I Live In Dreams
    Year:   1966
    Type:   release
    Role:   Main
    Format: Acetate, 7", Mono
    Label:  Orbit Music (2)

  London Boys
    Year:   1966
    Type:   release
    Role:   Main
    Format: Acetate, 7", Promo
    Label:  Pye Records

  I Dig Everything
    Year:   1966
    Type:   master
    Role:   Main
    Format: N/A
    Label:  N/A

  Can't Help Thinking About Me
    Year:   1966
    Type:   master
    Role:   Main
    Format: N/A
    Label:  N/A

  Rubber Band
    Year:   1966
    Type:   master
    Role:   Main
    Format: N/A
    Label:  N/A

  Do Anything You Say
    Year:   1966
    Type:   master
    Role:   Main
    Format: N/A
    Label:  N/A

  Sell Me A Coat
    Year:   1967
    Type:   release
    Role:   Main
    Format: Acetate, 7"
    Label:  TRO

  The Laughin

### 2a. Fetch a Specific Release for Genre/Style Detail

Individual release endpoints have the full `genres` and `styles` arrays.

In [7]:
# Pick the first release with a valid ID
first_release_id = releases['releases'][0]['id']
first_release_type = releases['releases'][0].get('type', 'release')

# Use 'masters' endpoint for master releases, 'releases' for regular
if first_release_type == 'master':
    release_detail = discogs_get(f"/masters/{first_release_id}")
else:
    release_detail = discogs_get(f"/releases/{first_release_id}")

print(f"Title:  {release_detail.get('title', 'N/A')}")
print(f"Year:   {release_detail.get('year', 'N/A')}")

# GENRES — broad categories
genres = release_detail.get('genres', [])
print(f"\nGenres: {genres}")

# STYLES — sub-genres (much more granular)
styles = release_detail.get('styles', [])
print(f"Styles: {styles}")

Title:  I Do Believe I Love You
Year:   1966

Genres: ['Rock', 'Pop']
Styles: ['Vocal']


### 2b. Collect Genres/Styles Across Multiple Releases

Let's see the full range of genres and styles for this artist.

In [8]:
all_genres = set()
all_styles = set()

# Fetch a few releases to collect genres/styles
# (Be mindful of rate limiting — 25 req/min)
for rel in releases['releases'][:5]:
    rel_id = rel['id']
    rel_type = rel.get('type', 'release')
    
    time.sleep(3)  # Respect rate limit
    
    try:
        if rel_type == 'master':
            detail = discogs_get(f"/masters/{rel_id}")
        else:
            detail = discogs_get(f"/releases/{rel_id}")
        
        all_genres.update(detail.get('genres', []))
        all_styles.update(detail.get('styles', []))
        print(f"  {detail.get('title', '?'):40s} | Genres: {detail.get('genres', [])} | Styles: {detail.get('styles', [])}")
    except Exception as e:
        print(f"  Error fetching {rel_id}: {e}")

print(f"\n--- Unique genres: {sorted(all_genres)}")
print(f"--- Unique styles: {sorted(all_styles)}")

  I Do Believe I Love You                  | Genres: ['Rock', 'Pop'] | Styles: ['Vocal']
  I Live In Dreams                         | Genres: ['Rock', 'Pop'] | Styles: ['Vocal', 'Acoustic']
  London Boys                              | Genres: ['Rock', 'Pop'] | Styles: ['Mod']
  I Dig Everything                         | Genres: ['Rock', 'Funk / Soul', 'Pop'] | Styles: ['Mod', 'Soul']
  Can't Help Thinking About Me             | Genres: ['Rock', 'Pop'] | Styles: ['Mod', 'Pop Rock']

--- Unique genres: ['Funk / Soul', 'Pop', 'Rock']
--- Unique styles: ['Acoustic', 'Mod', 'Pop Rock', 'Soul', 'Vocal']


## 3. A Band — The Beatles

Let's search for The Beatles and explore the group structure.

In [9]:
# Search for The Beatles
search_result = discogs_get("/database/search", params={"q": "The Beatles", "type": "artist", "per_page": 3})
pp(search_result['results'][:3])

[
  {
    "id": 82730,
    "type": "artist",
    "master_id": null,
    "master_url": null,
    "uri": "/artist/82730-The-Beatles",
    "title": "The Beatles",
    "thumb": "",
    "cover_image": "",
    "resource_url": "https://api.discogs.com/artists/82730"
  },
  {
    "id": 1385829,
    "type": "artist",
    "master_id": null,
    "master_url": null,
    "uri": "/artist/1385829-The-Beatles-Revival-Band",
    "title": "The Beatles Revival Band",
    "thumb": "",
    "cover_image": "",
    "resource_url": "https://api.discogs.com/artists/1385829"
  },
  {
    "id": 96263,
    "type": "artist",
    "master_id": null,
    "master_url": null,
    "uri": "/artist/96263-The-Tape-Beatles",
    "title": "The Tape-Beatles",
    "thumb": "",
    "cover_image": "",
    "resource_url": "https://api.discogs.com/artists/96263"
  }
]


In [10]:
# Note: search may require auth. If it fails, use the known Discogs ID.
# The Beatles Discogs ID is 82730
beatles_id = 82730

time.sleep(3)
beatles = discogs_get(f"/artists/{beatles_id}")

print(f"Name:    {beatles.get('name', 'N/A')}")
print(f"ID:      {beatles.get('id', 'N/A')}")

# Members
members = beatles.get('members', [])
print(f"\nMembers ({len(members)}):")
for m in members:
    print(f"  - {m['name']} (ID: {m['id']}, active: {m.get('active', '?')})")

# Profile
profile = beatles.get('profile', '')
print(f"\nProfile ({len(profile)} chars):")
print(profile[:1000])

Name:    The Beatles
ID:      82730

Members (6):
  - Paul McCartney (ID: 35301, active: True)
  - John Lennon (ID: 46481, active: True)
  - George Harrison (ID: 243955, active: True)
  - Richard Starkey (ID: 298525, active: True)
  - Pete Best (ID: 337031, active: False)
  - Stuart Sutcliffe (ID: 459513, active: False)

Profile (1136 chars):
Emerging from Liverpool, England in 1960, the Beatles were a seminal British group that evolved from rock/pop origins into pioneers of musical experimentation. The band solidified its iconic lineup in 1962, featuring [a46481] (vocals, rhythm guitar, harmonica, keyboards, percussion), [a35301] (vocals, bass, guitar, keyboards, percussion), [a243955] (lead guitar, vocals, keyboards), and [a259352] (drums, vocals, percussion) after signing a recording contract with EMI. This followed a brief period with Stuart Sutcliffe (bass, 1960-61) and Pete Best (drums, 1960-62).

Initially recognized for their Merseybeat style, the Beatles soon ventured into unc

## 4. Compare: MusicBrainz vs Discogs

Fill in your observations after running the cells above.

| Feature | MusicBrainz | Discogs | Winner for our KG |
|---|---|---|---|
| Artist basic info | name, type, country, gender, life-span | name, realname, name variations | MB (more structured) |
| Genre data | `tag-list` (flat, noisy, community-voted) | `genres` (broad) + `styles` (granular sub-genres) | **Discogs** (genre/subgenre hierarchy) |
| Artist biography | None | `profile` (free text) | **Discogs** (text source) |
| Band members | In `artist-rels` with instruments + dates | `members` list with active flag | **MB** (more detail) |
| Releases | Detailed with tracks, MBIDs | Detailed with formats, labels | Complementary |
| Relationships | Rich (influenced by, collaboration, etc.) | Limited (groups/members only) | **MB** (much richer) |
| Cross-references | URL rels to Discogs, Wikipedia, Wikidata | URLs to external sites | **MB** (more links) |
| Record labels | Via release data | Via release data (`label` field) | Comparable |

## 5. Cross-Reference Test

Can we go from MusicBrainz → Discogs using the URL rels?

In [11]:
# From Experiment 1, Bowie's Discogs URL was:
# https://www.discogs.com/artist/10263

discogs_url = "https://www.discogs.com/artist/10263"

# Extract the ID
discogs_id = int(discogs_url.split("/")[-1])
print(f"Extracted Discogs ID: {discogs_id}")

# Verify it works
time.sleep(3)
test = discogs_get(f"/artists/{discogs_id}")
print(f"Verified: {test['name']} (ID: {test['id']})")
print(f"\nCross-reference pipeline: MusicBrainz url-rels → extract Discogs ID → fetch Discogs data ✓")

Extracted Discogs ID: 10263
Verified: David Bowie (ID: 10263)

Cross-reference pipeline: MusicBrainz url-rels → extract Discogs ID → fetch Discogs data ✓


## Next Steps

- **Experiment 3**: Wikipedia API — fetch article text for NER/LLM extraction
- **Experiment 4**: Cross-reference all three sources for the same entity
- **Key decision**: Use MusicBrainz as primary structured source, Discogs for genre hierarchy + biography text